In [1]:
'THE FOLLOWING CODE IS THE PEAK IDENTIFICATION METHOD USED TO FILTER ANTICORRELATED EVENTS' 
'WHICH DO NOT EXHIBITS A WAVE BEHAVIOUR'

'ie linear regression and prominence method'

'We also filter the negative values of the plasma density'
#CHECKED

'We also filter the negative values of the plasma density'

## Functions

In [2]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method conzzzzt of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCA2(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    indexA=zx2['Index']
    
    #Empty Dataframes
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,


def PCA2(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    #print(X)
    check_for_nan_1 = X['Bx'].isnull()
    check_for_nan_2 = X['By'].isnull()
    check_for_nan_3 = X['Bz'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True or check_for_nan_2[i]==True or check_for_nan_3[i]==True :
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #print(data_copy)
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        covariance_matrix=data_copy.cov()
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#---------------------------------------------------------------------------------------------------------------

## Import Data

In [3]:
#import

#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2016/March_2016/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2016/February_2016/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2016/January_2016/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2014/November_2014/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2014/December_2014/MM_final_table_anticorrelation"

#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/December_2015/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/November_2015/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/October_2015/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/September_2015/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/August_2015/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/July_2015/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/June_2015/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/May_2015/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/April_2015/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/March_2015/MM_final_table_anticorrelation"
folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/MM_final_table_anticorrelation"
#folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/December_2015/MM_final_table_anticorrelation"
#---------------------------------------------------------
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2016/March_2016/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2016/February_2016/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2016/January_2016/DATA"

#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2014/November_2014/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2014/December_2014/DATA"

#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/December_2015/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/November_2015/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/October_2015/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/September_2015/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/August_2015/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/July_2015/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/June_2015/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/May_2015/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/April_2015/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/March_2015/DATA"
data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA"
#data_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/December_2015/DATA"
#---------------------------------------------------------
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2016/marzo"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2016/febrero"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero"

#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2014/noviembre"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2014/diciembre"

#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/diciembre"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/noviembre"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/octubre"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/septiembre"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/agosto"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/julio"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/junio"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/mayo"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/abril"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/marzo"
plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero"
#plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/diciembre"

In [4]:
#IMPORTING DATA:

directory= os.scandir(folder)
list_of_files=get_directory(folder)

newlist=[]
for item in list_of_files:
    newlist.append(folder+'/'+str(item))
print(newlist)


MM_final_table=[]
for item in newlist:
    #Importing data
    titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson']
    data= pd.read_table(str(item), header=None, names=titles, sep='\t')
    data2=data.iloc[np.arange(1,len(data)),:] #delete the first row
    data2=data2.reset_index(drop=True)
    MM_final_table.append(data2)
       

['C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/MM_final_table_anticorrelation/MM_final_table_anticorrelation_20150201', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/MM_final_table_anticorrelation/MM_final_table_anticorrelation_20150202', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/MM_final_table_anticorrelation/MM_final_table_anticorrelation_20150203', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/MM_final_table_anticorrelation/MM_final_table_anticorrelation_20150204', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/MM_final_table_anticorrelation/MM_final_table_anticorrelation_20150205', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/MM_final_table_anticorrelation/MM_final_table_anticorrelation_20150206', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/MM_final_table_anticorrelation/MM_final_table_anti

In [5]:
list_of_data=get_directory(data_folder)

#List with the paths
newlist2=[]
for item in list_of_data:
    newlist2.append(data_folder+'/'+str(item))
print(newlist2)

MM_final_table2=[] 

for i in np.arange(0, len(newlist2)):
        title2=['array2','array3','array4','array5','array6','array8','array9','array10','array11','array12','array13','array14','array15', 'year','month','day']
        path= str(newlist2[i])
        data1= pd.read_table(path, header=None, names=title2, sep='\t')
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        data2=data2.reset_index(drop=True)
        for j in np.arange(0,len(data2)):
            if type(data2['array2'][j])==str:
                data2['array2'][j]=float(data2['array2'][j])
           
        MM_final_table2.append(data2)     

['C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA/DATA_20150201', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA/DATA_20150202', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA/DATA_20150203', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA/DATA_20150204', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA/DATA_20150205', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA/DATA_20150206', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA/DATA_20150207', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA/DATA_20150208', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA/DATA_20150209', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/February_2015/DATA/DATA_20150210', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/2015/

C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\1255548653.py:14: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  data1= pd.read_table(path, header=None, names=title2, sep='\t')
C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\1255548653.py:14: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  data1= pd.read_table(path, header=None, names=title2, sep='\t')
C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\1255548653.py:14: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  data1= pd.read_table(path, header=None, names=title2, sep='\t')
C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\1255548653.py:14: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15) have mixed types. Specify dtype option on import or set low_

In [6]:
years=[]
months=[]
days=[]
array2=[] #bmodulus


for i in np.arange(0,len(newlist2)):
    #convertedArray = sampleArray.astype(np.float)
    array2.append(MM_final_table2[i]['array2'].astype(np.float))
    years.append(int(float(MM_final_table2[i]['year'][0])))
    months.append(int(float(MM_final_table2[i]['month'][0])))
    days.append(int(float(MM_final_table2[i]['day'][0])))   

C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\170287525.py:9: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  array2.append(MM_final_table2[i]['array2'].astype(np.float))


In [7]:
#HIGH RESOLUTION PLASMA FOLDER
#---------------------------------------------------------

list_of_files_plasma=get_directory(plasma_folder)

#List with the paths
newlist=[]
for item in list_of_files_plasma:
    newlist.append(plasma_folder+'/'+str(item))
print(newlist)  

year_plasma=[]
month_plasma=[]
day_plasma=[]
list_of_plasma=[] #List of arrays with plasma densities

for i in np.arange(0, len(newlist)):
        title2=['t_utc','?','npl','??','???','????']
        path= str(newlist[i])
        data1= pd.read_table(path, header=None, names=title2, sep=',', engine='python', parse_dates=['t_utc'])
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        data2=data2.reset_index(drop=True)
        data2=data2[['t_utc','npl']]
        
        #--------------------------------------------------------------------------
        #Saving the dates
        path_time= pd.to_datetime(data2['t_utc']) #data2['t_utc']
        #It is needed to obtain the year, month and day of an specific path
        q=path_time.dt.year
        qq=path_time.dt.month
        qqq=path_time.dt.day
        year=q[0]
        month=qq[0]
        day=qqq[0]
        
        year_plasma.append(year)
        month_plasma.append(month)
        day_plasma.append(day)
        #------------------------------------------------------------------------------
        #Resample
        data2['index'] = pd.to_datetime(data2['t_utc'])
        data2.set_index('index', inplace=True)
        data2=data2.resample('1S').mean()

        data2['t_utc'] = pd.to_datetime(data2.index.values)
        data2= data2.reset_index()
        #time_lap=pd.to_datetime(data2['t_utc'])
        #---------------------------------------------------------------------------------

        #Filling the data gaps
        data2.t_utc = data2.t_utc.dt.round('1s') #round to one second for simplicity
        if data2.shape[0] < 86400: # if the number of datapoints is lower than one day:
            #print('Data gaps detected, padding array....')
            data3 = data2.rename(index=(data2['t_utc']-data2.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
            data3 = data3.reindex(range(0,86400)) # now we just fill in the missing values
            newt = pd.date_range(start = data2.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
            data3['t_utc'] = newt # now fill in the times so there is no NaT
            data2 = data3 
            del(data3)
        elif data2.shape[0] > 86400:
            error('Data file is too long, probably need to debug the code again....')
            print('Done\n')
        
        list_of_plasma.append(data2['npl'])  

['C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150202_000212_805_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150204_000212_805_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150206_000108_805_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150207_000001_805_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150208_000212_805_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150210_000108_805_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150211_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150212_000212_805_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150213_160213_805_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150214_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150215_000005_805_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2015/febrero/LAP_20150216_000213_805_NEL.TAB', 'C:/Users/Ari

C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\1187717895.py:44: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  data2=data2.resample('1S').mean()
C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\1187717895.py:44: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  data2=data2.resample('1S').mean()
C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\1187717895.py:44: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  data2=data2

## Dataframe creation 

In [8]:

final_events=[] #list with dataframes (bmodulus, npl) for each day with resample
final_events_without_resample=[]
for i in np.arange(0,len(MM_final_table)): #for every day   
    
    if len(MM_final_table[i])==0: #If there are not MM mode candidates
        #listt.append(pd.DataFrame([]))
        #list2.append(pd.DataFrame([]))
        final_events.append(pd.DataFrame([]))
        final_events_without_resample.append(pd.DataFrame([]))
        
    else: 
        
        idx = pd.date_range(str(years[i])+'-'+str(months[i])+'-'+str(days[i]), periods=86400, freq="S")
        path_time=pd.to_datetime(idx)

        #Creating a new dataframe in order to establish the anticorrelation
        a2={ 'Dates_and_Hours': idx, 'bmodulus': array2[i]}
        data_correlation2= pd.DataFrame(a2,columns = ['Dates_and_Hours', 'bmodulus'])
        data_correlation2['index']=path_time
        #-------------------------------------------------
        #Finding the correct npl list for the day
        if days[i] in day_plasma:
            indexx=[]
            for k in np.arange(0,len(day_plasma)):
                if days[i]==day_plasma[k]:
                    indexx.append(k)
            
            LAP_plasma=list_of_plasma[indexx[0]]
        else:
            LAP_plasma=[]    
        #-----------------------------------------------------------------------------------
        #Adding the plasma density to the dataframe
        if len(LAP_plasma)!=0: #If we have high density plasma data
            data_correlation2['npl']=LAP_plasma
        else:
            new_col = np.empty((array16[i].shape[0],1)) #if not it is filled with nan values
            new_col.fill(np.nan)
            data_correlation2['npl']=new_col
            
        data_correlation2.set_index('index', inplace=True)       
        #-----------------------------------------------------------------------------------
        'Here we have data for each day' 
        #ITERATION FOR EVERY EVENT
        
        events_to_correlate=[] #List of dataframes that includes bmodulus and npl for every time interval WITHOUT RESAMPLE
        events_to_correlate2=[] #WITH RESAMPLE 
        for j in np.arange(0,len(MM_final_table[i])):
            start=str(MM_final_table[i]['Beginning'][j])
            start_date = datetime.datetime.strptime(start, '%Y-%m-%d %H:%M:%S')
            start_date2=start_date.strftime('%H:%M:%S') #Time with the format needed
            end=str(MM_final_table[i]['End'][j])
            end_date=datetime.datetime.strptime(end, '%Y-%m-%d %H:%M:%S')
            end_date2=end_date.strftime('%H:%M:%S') #Time with the format needed
            #Choosing the time
            df1 = data_correlation2.between_time(start_time=start_date2, end_time=end_date2) #Dataframe with the MM event data
            #---------------------
            #Resample
            df2=df1.copy()
            df2['index2'] = pd.to_datetime(data_correlation2['Dates_and_Hours'])
            df2.set_index('index2', inplace=True)
            
            df3=df2.copy()
            df3['t_utc'] = pd.to_datetime(df3.index.values)
            
            df2=df2.resample('3S').mean()
            df2['t_utc'] = pd.to_datetime(df2.index.values)
            df2= df2.reset_index()
            #---------------------
            events_to_correlate.append(df3)
            events_to_correlate2.append(df2)
        final_events.append(events_to_correlate2)
        final_events_without_resample.append(events_to_correlate)
 #-----------------------------------------------------          


C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\2630360347.py:65: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df2=df2.resample('3S').mean()
C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\2630360347.py:65: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df2=df2.resample('3S').mean()
C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\2630360347.py:65: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df2=df2.resample('3

## Iteration

In [14]:
#If you want to plot the events without resample you need to change the name of final_events_without_resample to final_events only

#for i in np.arange(0,len(final_events)):
listt=[] #Array with the final events using the linear regression method
listt2=[] #Array with the final events using the prominence method
#for i in np.arange(17,18): 

for i in np.arange(0,len(MM_final_table)):  
#for i in np.arange(6,12):  #FIRST PLOT  
#for i in np.arange(6,12):  #SECOND PLOT  
    print('i:', i)
    MM_final_table_copy=MM_final_table[i].copy()
    MM_final_table_copy2=MM_final_table[i].copy()
    print(MM_final_table_copy)
    for j in np.arange(0,len(final_events[i])):
        
        print('j:',j)
        quantity1=final_events[i][j]['t_utc']
        quantity2=final_events[i][j]['npl']
        print('quantity2')
        print(quantity2)
        quantity3=final_events[i][j]['bmodulus']

        from sklearn.linear_model import LinearRegression
        #------------------------------------------------------------------------------
        #LINEAR REGRESSIONS
        regressor = LinearRegression()
       
        count=[]
        count2=[]
        for k in np.arange(1,len(quantity2)+1):
            count.append([k])
            count2.append(k)
       
        #IF WE HAVE NAN VALUES IN THE MIDDLE
        a1=0
        for item in quantity2:
            if np.isnan(item)==True:
                a1=1
        if a1==1:
            ajuste1= quantity2.interpolate()
        else:
            ajuste1=quantity2.copy()
        
        #IF WE HAVE A NAN VALUE AT THE BEGINNING
        xx=[]
        if np.isnan(ajuste1[0])==True:
            xx.append(1)
            ajuste1.drop(0, axis=0, inplace=True)
            ajuste1=ajuste1.reset_index(drop=True)
            count=[]
            count2=[]
            for k in np.arange(1,len(quantity2)):
                count.append([k])
                count2.append(k)
        
        #IF WE HAVE A SECOND NAN VALUE AT THE BEGINNING
        xxsecond=[]
        if np.isnan(ajuste1[0])==True:
            xxsecond.append(1)
            ajuste1.drop(0, axis=0, inplace=True)
            ajuste1=ajuste1.reset_index(drop=True)
            count=[]
            count2=[]
            for k in np.arange(1,len(quantity2)-1):
                count.append([k])
                count2.append(k)
        
        #IF WE HAVE A NAN VALUE AT THE END 
        #xx2=[]
        #if np.isnan(ajuste1[len(ajuste1)-1])==True:
         #   xx2.append(1)
          #  a1=len(count2)
           # ajuste1.drop(len(ajuste1)-1, axis=0, inplace=True)
            #ajuste1=ajuste1.reset_index(drop=True)
            #count=[]
            #count2=[]
            #for k in np.arange(1,len(a1)):
             #   count.append([k])
              #  count2.append(k)
        
        #IF WE HAVE A SECOND NAN VALUE AT THE END 
        #xx2second=[]
        #if np.isnan(ajuste1[len(ajuste1)-1])==True:
         #   xx2second.append(1)
         #   a1=len(count2)
          #  ajuste1.drop(len(ajuste1)-1, axis=0, inplace=True)
           # ajuste1=ajuste1.reset_index(drop=True)
           # count=[]
           # count2=[]
           # for k in np.arange(1,len(a1)):
            #    count.append([k])
             #   count2.append(k)        
        
        q=0
        for item in ajuste1:
            if np.isnan(item)==True:
                q=q+1
        
        #If we have more nan values at the beginning after removing 2 nans
        if q>=1:
            ajuste1final=[]
            score1=1
        else:
            z1=regressor.fit(count,ajuste1)
            score1=z1.score(count, ajuste1)
            m = z1.coef_[0]
            b = z1.intercept_
            ajuste1final=[]
            for item in count2:
                ajuste1final.append(m*item+b)

            #If we deleted one at the beginning
            if len(xx)==1:
                ajuste1final.insert(0, np.nan)
            
            #If we deleted 2 at the beginning
            if len(xxsecond)==1:
                ajuste1final.insert(0, np.nan)    
        
                               
        regressor2 = LinearRegression()
        
        countt=[]
        countt2=[]
        for l in np.arange(1,len(quantity3)+1):
            countt.append([l])
            countt2.append(l)
       
        #IF WE HAVE NAN VALUES AT THE MIDDLE
        a11=0
        for item in quantity3:
            if np.isnan(item)==True:
                a11=1
        if a11==1:
            ajuste2= quantity3.interpolate()
        else:
            ajuste2=quantity3.copy()
            
            
        #IF WE HAVE A NAN VALUE AT THE BEGINNING
        xxx=[]
        if np.isnan(ajuste2[0])==True:
            xxx.append(1)
            ajuste2.drop(0, axis=0, inplace=True)
            ajuste2=ajuste2.reset_index(drop=True)
            countt=[]
            countt2=[]
            for k in np.arange(1,len(quantity3)):
                countt.append([k])
                countt2.append(k)
        
        #IF WE HAVE A SECOND NAN VALUE AT THE BEGINNING
        xxxsecond=[]
        if np.isnan(ajuste2[0])==True:
            xxxsecond.append(1)
            ajuste2.drop(0, axis=0, inplace=True)
            ajuste2=ajuste2.reset_index(drop=True)
            countt=[]
            countt2=[]
            for k in np.arange(1,len(quantity2)-1):
                countt.append([k])
                countt2.append(k)
        
        #IF WE HAVE A NAN VALUE AT THE END 
        #xxx2=[]
        #if np.isnan(ajuste2[len(ajuste2)-1])==True:
         #   xxx2.append(1)
          #  a1=len(countt2)
          #  ajuste2.drop(len(ajuste2)-1, axis=0, inplace=True)
           # ajuste2=ajuste2.reset_index(drop=True)
            
            #countt=[]
            #countt2=[]
            #for k in np.arange(1,len(a1)):
             #   countt.append([k])
              #  countt2.append(k)    
        #IF WE HAVE A SECOND NAN VALUE AT THE END 
        #xxx2second=[]
        #if np.isnan(ajuste2[len(ajuste2)-1])==True:
         #   xxx2second.append(1)
          #  a1=len(countt2)
           # ajuste2.drop(len(ajuste2)-1, axis=0, inplace=True)
           # ajuste2=ajuste2.reset_index(drop=True)
           # countt=[]
           # countt2=[]
           # for k in np.arange(1,len(a1)):
            #    countt.append([k])
             #   countt2.append(k) 
        
        
        q2=0
        for item in ajuste2:
            if np.isnan(item)==True:
                q2=q2+1
        
        #If we have more nan values at the beginning:
        if q2>=1:
            ajuste2final=[]
            score2=1
        else:
            
            z2=regressor2.fit(countt,ajuste2 )
            score2=z2.score(countt, ajuste2)
            mm = z2.coef_[0]
            bb = z2.intercept_
            ajuste2final=[]
            for item in countt2:
                ajuste2final.append(mm*item+bb)
            
            #If we deleted one at the beginning:
            if len(xxx)==1:
                ajuste2final.insert(0, np.nan)
            #If we deleted two at the beginning:
            if len(xxxsecond)==1:
                ajuste2final.insert(0, np.nan)   
        
        #------------------------------------------------------------------------------
        
        from scipy.signal import find_peaks
        

#-------------------------
#PROMINENCE METHOD 
         
        indicess = find_peaks(quantity2, height=0.00002, prominence=(None, 4000), width=(0.000002, 4000),  rel_height=1)
        pir=[]
        for m in np.arange(0,len(indicess[1]['peak_heights'])):
            pir.append([indicess[0][m],indicess[1]['prominences'][m],indicess[1]['peak_heights'][m],abs(indicess[1]['right_ips'][m]-indicess[1]['left_ips'][m])])
        
        indicess2 = find_peaks(-1*quantity2, height=-400000, prominence=(None, 4000), width=(0.000002, 4000),  rel_height=1) 
        pir2=[]
        for m in np.arange(0,len(indicess2[1]['peak_heights'])):
            pir2.append([indicess2[0][m],indicess2[1]['prominences'][m],indicess2[1]['peak_heights'][m],abs(indicess2[1]['right_ips'][m]-indicess2[1]['left_ips'][m])]) 
        
        indicess3 = find_peaks(quantity3, height=0.00002, prominence=(None, 4000), width=(0.000002, 4000),  rel_height=1) 
        pir3=[]
        for m in np.arange(0,len(indicess3[1]['peak_heights'])):
            pir3.append([indicess3[0][m],indicess3[1]['prominences'][m],indicess3[1]['peak_heights'][m],abs(indicess3[1]['right_ips'][m]-indicess3[1]['left_ips'][m])])
        
        indicess4 = find_peaks(-1*quantity3, height=-400000, prominence=(None, 4000), width=(0.000002, 4000),  rel_height=1)
        pir4=[]
        for m in np.arange(0,len(indicess4[1]['peak_heights'])):
            pir4.append([indicess4[0][m],indicess4[1]['prominences'][m],indicess4[1]['peak_heights'][m],abs(indicess4[1]['right_ips'][m]-indicess4[1]['left_ips'][m])])
        
        #Keep the peaks with the highest prominences
        if len(pir)!=0:
            max11=max(pir, key=lambda x: x[1]) 
        else:
            max11=[]
            
        if len(pir2)!=0:
            max22=max(pir2, key=lambda x: x[1]) 
        else:
            max22=[]
        
        if len(pir3)!=0:
            max33=max(pir3, key=lambda x: x[1]) 
        else:
            max33=[]
            
        if len(pir4)!=0:
            max44=max(pir4, key=lambda x: x[1]) 
        else:
            max44=[]    
  
        FINALMAXMINPROMINENCE=[] #MAXIMUM NPL FIRST, THEN MIN BMODULUS
        FINALMINMAXPROMINENCE=[] #MINIMUM NPL FIRST, THEN MAX BMODULUS
        
        
        if len(max11)!=0 and len(max44)!=0:
            #if the time difference between the peaks are lower than the half of maximum between the width of the peaks / It is multiplied by 3/2 because each data has 3 second between each other and it is divided by 2 becase we want half of the peak
            if (abs(quantity1[max11[0]].second-quantity1[max44[0]].second)<=(max(max11[3],max44[3])*(3/2)) and quantity1[max11[0]].minute-quantity1[max44[0]].minute==0) or ((60-(abs(quantity1[max11[0]].second-quantity1[max44[0]].second)))<=(max(max11[3],max44[3])*(3/2)) and abs(quantity1[max11[0]].minute-quantity1[max44[0]].minute)==1):
                FINALMAXMINPROMINENCE.append([max11,max44])
        
        if len(max22)!=0 and len(max33)!=0:
            #if the time difference between the peaks are lower than the half of maximum between the width of the peaks / It is multiplied by 3/2 because each data has 3 second between each other and it is divided by 2 becase we want half of the peak
            if (abs(quantity1[max22[0]].second-quantity1[max33[0]].second)<=(max(max22[3],max33[3])*(3/2)) and quantity1[max22[0]].minute-quantity1[max33[0]].minute==0) or ((60-(abs(quantity1[max22[0]].second-quantity1[max33[0]].second)))<=(max(max22[3],max33[3])*(3/2)) and abs(quantity1[max22[0]].minute-quantity1[max33[0]].minute)==1):
                FINALMINMAXPROMINENCE.append([max22,max33])
    
#-------------------------
#LINEAR REGRESSION METHOD

        indices = find_peaks(quantity2)[0] 
        indices2 = find_peaks(-1*quantity2)[0] 
 
        indices3 = find_peaks(quantity3)[0] 
        indices4 = find_peaks(-1*quantity3)[0] 
        
        #Variations
        list1=[]
        list2=[]
        list3=[]
        list4=[]
        
        if len(ajuste1final)==0:
            max1=[]
        else:
            for item in indices:
                if np.isnan(ajuste1final[item])==True: #We do this if there is a peak just where there was a nan value at the left/right corner of the linear regr.
                    list1.append([0,0,0])
                else:
                    delta= abs(quantity2[item]-ajuste1final[item])
                    list1.append([delta,item,quantity2[item]]) #[delta, index, value]
        
        if len(ajuste1final)==0:
            max2=[]
        else:    
            for item in indices2:
                if np.isnan(ajuste1final[item])==True:
                    list2.append([0,0,0])
                else:
                    delta= abs(quantity2[item]-ajuste1final[item])
                    list2.append([delta,item,quantity2[item]])
        
        if len(ajuste2final)==0:
            max3=[]
        else:
            for item in indices3:
                if np.isnan(ajuste2final[item])==True:
                    list3.append([0,0,0])
                else:
                    delta= abs(quantity3[item]-ajuste2final[item])
                    list3.append([delta,item,quantity3[item]])
        
        if len(ajuste2final)==0:
            max4=[]
        else:
            for item in indices4:
                if np.isnan(ajuste2final[item])==True:
                    list4.append([0,0,0])
                else:
                    delta= abs(quantity3[item]-ajuste2final[item])
                    list4.append([delta,item,quantity3[item]])
        #------------------------
        #This returns the index of the max and min of each function  
        if len(list1)!=0:
            max1=max(list1, key=lambda x: x[0]) 
        else:
            max1=[]
        
        if len(list2)!=0:
            max2=max(list2, key=lambda x: x[0])
        else:
            max2=[]
        
        if len(list3)!=0:
            max3=max(list3, key=lambda x: x[0])
        else:
            max3=[]
        
        if len(list4)!=0:
            max4=max(list4, key=lambda x: x[0])
        else:
            max4=[]
        
        FINALMAXMIN=[] #MAXIMUM NPL GOES FIRST, THEN MIN BMODULUS
        FINALMINMAX=[] #MINIMO NPL GOES FIRST, THEN MAX BMODULUS
        
        secondsthreshold=5
        if len(max1)!=0 and len(max4)!=0:
            if (abs(quantity1[max1[1]].second-quantity1[max4[1]].second)<secondsthreshold and quantity1[max1[1]].minute-quantity1[max4[1]].minute==0) or ( (60-secondsthreshold)<=abs(quantity1[max1[1]].second-quantity1[max4[1]].second)<=59 and quantity1[max1[1]].minute-quantity1[max4[1]].minute==1):
                FINALMAXMIN.append([max1,max4])
        
        if len(max2)!=0 and len(max3)!=0:
            if (abs(quantity1[max2[1]].second-quantity1[max3[1]].second)<secondsthreshold and quantity1[max2[1]].minute-quantity1[max3[1]].minute==0) or ( (60-secondsthreshold)<=abs(quantity1[max2[1]].second-quantity1[max3[1]].second)<=59 and quantity1[max2[1]].minute-quantity1[max3[1]].minute==1):
                FINALMINMAX.append([max2,max3])
        
            
        'SELECTION'
        
        #SELECTION FOR THE LINEAR REGRESSION METHOD
        #if score1>0.7 or score2>0.7 or (score1<=0.7 and score2<=0.7 and len(FINALMAXMIN)==0 and len(FINALMINMAX)==0):
         #   MM_final_table_copy['Beginning'][j]=np.nan
          #  MM_final_table_copy['End'][j]=np.nan
        
        #SELECTION FOR THE LINEAR PROMINENCE METHOD
        #if score1>0.7 or score2>0.7 or (score1<=0.7 and score2<=0.7 and len(FINALMAXMINPROMINENCE)==0 and len(FINALMINMAXPROMINENCE)==0):
         #   MM_final_table_copy2['Beginning'][j]=np.nan
          #  MM_final_table_copy2['End'][j]=np.nan 
        
        'SELECTION II'
        
        print('MINIMO')
        print(np.nanmin(quantity2))
        #SELECTION FOR THE LINEAR REGRESSION METHOD
        if score1>0.7 or score2>0.7 or (score1<=0.7 and score2<=0.7 and len(FINALMAXMIN)==0 and len(FINALMINMAX)==0) or np.nanmin(quantity2)<0: #MODIFICAR CON NANMIN
            MM_final_table_copy['Beginning'][j]=np.nan
            MM_final_table_copy['End'][j]=np.nan
        
        #SELECTION FOR THE LINEAR PROMINENCE METHOD
        if score1>0.7 or score2>0.7 or (score1<=0.7 and score2<=0.7 and len(FINALMAXMINPROMINENCE)==0 and len(FINALMINMAXPROMINENCE)==0) or np.nanmin(quantity2)<0:
            MM_final_table_copy2['Beginning'][j]=np.nan
            MM_final_table_copy2['End'][j]=np.nan 
        
        
        #PLOT
        
        #----------------------------------
        ysize=10
        xsize=10
        zz=[quantity1[j] for j in indices]
        zzz=[quantity2[j] for j in indices]
        zzzz=[quantity1[j] for j in indices2]
        zzzzz=[quantity2[j] for j in indices2]
        
        zz2=[quantity1[j] for j in indices3]
        zzz2=[quantity3[j] for j in indices3]
        zzzz2=[quantity1[j] for j in indices4]
        zzzzz2=[quantity3[j] for j in indices4]
        
        #Plasma density
        
        #PLOT 30 ES UNO DE LOS EVENTOS
        #PLOT 49 ES UNO DE LOS EVENTOS
        #plot 62
        #plot 71
        
        plt.figure()
        ax7=plt.subplot(211)
        figsize=(20,4)
        plt.plot(quantity1, round(quantity2), '-.',c='black', label='LAP plasma density')  
        #plt.title(str(quantity1[0])+str(quantity1[len(quantity1)-1]))
        
        if len(ajuste1final)!=0:
            if score1<0.7:
                plt.plot(quantity1, ajuste1final, '-',c='purple', label='Linear Regression')
            else:
                plt.plot(quantity1, ajuste1final, '-',c='blue', label='Linear Regression') 
        
        plt.plot(zz, zzz, 'x',c='black') #Plots all minima and maxima 
        plt.plot(zzzz, zzzzz, 'x',c='black')
        
        if len(FINALMAXMINPROMINENCE)!=0:
            plt.plot(quantity1[FINALMAXMINPROMINENCE[0][0][0]],quantity2[FINALMAXMINPROMINENCE[0][0][0]], 'x', c='red')
        if len(FINALMINMAXPROMINENCE)!=0:
            plt.plot(quantity1[FINALMINMAXPROMINENCE[0][0][0]],quantity2[FINALMINMAXPROMINENCE[0][0][0]], 'x', c='red')
        
        #if len(max11)!=0:
          #  plt.plot(quantity1[max11[0]],max11[2], '.', c='red')
        #if len(max22)!=0:
           # plt.plot(quantity1[max22[0]],-1*max22[2], '.', c='red')
        
        plt.ylabel('n ($cm^{-3}$)',fontsize=10)        
        ax7.tick_params(axis="y", labelsize=ysize)
        ax7.yaxis.set_label_position("right")
        ax7.yaxis.tick_right()
        #plt.xlabel('Time (UTC)',fontsize=10)
        xfmt = mdates.DateFormatter("%H:%M:%S")
        ax7.xaxis.set_major_formatter(xfmt)
        #plt.setp(ax7.get_xticklabels(), fontsize=6)
        plt.setp(ax7.get_xticklabels(), visible=False)
        #plt.legend()

        plt.title('R2 npl='+str(round(score1, 2))+' and'+' R2 bmodulus='+str(round(score2, 2)),fontsize=15)
        
        #ax7.get_yaxis().set_label_coords(1.082,0.5)
        ax7.get_yaxis().set_label_coords(1.092,0.5)

        
        #bmodulus
        ax2=plt.subplot(212, sharex=ax7)
        figsize=(20,4)
        plt.plot(quantity1, round(quantity3), '-.',c='black', label='bmodulus')
        plt.plot(zz2, zzz2, 'x',c='black')
        plt.plot(zzzz2, zzzzz2, 'x',c='black')
        
        if len(ajuste2final)!=0:
            if score2<0.7:
                plt.plot(quantity1, ajuste2final, '-',c='purple', label='Linear Regression')
            else:
                plt.plot(quantity1, ajuste2final, '-',c='blue', label='Linear Regression')

        if len(FINALMAXMINPROMINENCE)!=0:
            plt.plot(quantity1[FINALMAXMINPROMINENCE[0][1][0]],quantity3[FINALMAXMINPROMINENCE[0][1][0]], 'x', c='red')
        if len(FINALMINMAXPROMINENCE)!=0:
            plt.plot(quantity1[FINALMINMAXPROMINENCE[0][1][0]],quantity3[FINALMINMAXPROMINENCE[0][1][0]], 'x', c='red')
        
        #if len(max33)!=0:
         #   plt.plot(quantity1[max33[0]],max33[2], '.', c='red')
        #if len(max44)!=0:
         #   plt.plot(quantity1[max44[0]],-1*max44[2], '.', c='red')   
        
        plt.ylabel('|B| (nT)',fontsize=10)
        ax2.tick_params(axis="y", labelsize=ysize)
        ax2.yaxis.set_label_position("right")
        ax2.yaxis.tick_right()
        plt.xlabel('Time (UTC)',fontsize=10)
        xfmt = mdates.DateFormatter("%H:%M:%S")
        ax7.xaxis.set_major_formatter(xfmt)
        plt.setp(ax2.get_xticklabels(), fontsize=xsize)
        
        ax2.get_yaxis().set_label_coords(1.092,0.5)
        #plt.legend()
        
        
#---------------------------------------------  
    #DROPPING CANDIDATES LINEAR REGRESSION METHOD
    
    for j in np.arange(0,len(MM_final_table_copy)):
        if pd.isnull(MM_final_table_copy['Beginning'][j])==True and pd.isnull(MM_final_table_copy['End'][j])==True:
            MM_final_table_copy.drop(j, axis=0, inplace=True)
        
    MM_final_table_copy=MM_final_table_copy.reset_index(drop=True)
    #-<------------------------------------------------- 
    if len(MM_final_table_copy)==0:
        listt.append(pd.DataFrame([]))
    else:    
        listt.append(MM_final_table_copy)
        
         
#---------------------------------------------  
    #DROPPING CANDIDATES PROMINENCE METHOD
    
    for j in np.arange(0,len(MM_final_table_copy2)):
        if pd.isnull(MM_final_table_copy2['Beginning'][j])==True and pd.isnull(MM_final_table_copy2['End'][j])==True:
            MM_final_table_copy2.drop(j, axis=0, inplace=True)
    
    print('FINAL RESULT')
    print(MM_final_table_copy2)
    MM_final_table_copy2=MM_final_table_copy2.reset_index(drop=True)
    #--------------------------------------------------  
    if len(MM_final_table_copy2)==0:
        listt2.append(pd.DataFrame([]))
    else:    
        listt2.append(MM_final_table_copy2)        

        
#end        

i: 0
Empty DataFrame
Columns: [Beginning, End, Index1, Index2, Pearson]
Index: []
FINAL RESULT
Empty DataFrame
Columns: [Beginning, End, Index1, Index2, Pearson]
Index: []
i: 1
             Beginning                  End   Index1   Index2  \
0  2015-02-02 11:37:38  2015-02-02 11:38:08  41873.0  41873.0   

              Pearson  
0  -0.842309918557881  
j: 0
quantity2
0     67.705886
1     67.705882
2     67.705821
3     64.154789
4     57.051817
5     53.498972
6     42.840907
7     46.390595
8     46.388241
9     46.385415
10    46.382589
Name: npl, dtype: float64
MINIMO
42.840906999999994
FINAL RESULT
Empty DataFrame
Columns: [Beginning, End, Index1, Index2, Pearson]
Index: []
i: 2
Empty DataFrame
Columns: [Beginning, End, Index1, Index2, Pearson]
Index: []
FINAL RESULT
Empty DataFrame
Columns: [Beginning, End, Index1, Index2, Pearson]
Index: []
i: 3
             Beginning                  End   Index1   Index2  \
0  2015-02-04 13:17:39  2015-02-04 13:18:10  47874.0  47875.0   
1  2

C:\Users\Ariel\AppData\Local\Temp\ipykernel_15896\331035151.py:419: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure()


j: 2
quantity2
0     55.508273
1     68.333539
2     29.856203
3      4.204431
4    -21.445984
5      4.204487
6    -12.895643
7     12.754579
8      4.204603
9    -47.095214
10   -47.094945
Name: npl, dtype: float64
MINIMO
-47.095214
j: 3
quantity2
0    -21.668342
1     -5.058302
2      3.247860
3      3.247891
4      3.247922
5      3.247960
6     -9.210937
7      3.248022
8     11.553925
9     28.165619
10    28.165557
Name: npl, dtype: float64
MINIMO
-21.668342
FINAL RESULT
Empty DataFrame
Columns: [Beginning, End, Index1, Index2, Pearson]
Index: []
i: 8
Empty DataFrame
Columns: [Beginning, End, Index1, Index2, Pearson]
Index: []
FINAL RESULT
Empty DataFrame
Columns: [Beginning, End, Index1, Index2, Pearson]
Index: []
i: 9
              Beginning                  End   Index1   Index2  \
0   2015-02-10 00:06:16  2015-02-10 00:07:08    391.0    413.0   
1   2015-02-10 00:08:00  2015-02-10 00:08:51    495.0    516.0   
2   2015-02-10 00:15:09  2015-02-10 00:15:45    924.0    930.0   

In [10]:
print(round(23.5))

24


In [11]:
print(len(listt2))
#print(listt2)

28


In [12]:
print(days)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]


In [13]:
done

NameError: name 'done' is not defined

## Saving files

In [ ]:
import csv

#SAVING THE TABLES IN A CSV FILE
for i in np.arange(0,len(listt)):
    #listt[i].to_csv('MM_final_table_anticorrelation_linear_regression_method'+'_'+str(years[i])+str(months[i])+str(days[i]), index=False, sep='\t')
    listt2[i].to_csv('MM_final_table_with_pearson_prominence_method'+'_'+str(years[i])+str(months[i])+str(days[i]), index=False, sep='\t')